# 1d example of embedding restriction

We show that a non-augmented model of 1d input to 1d output cannot approximate the function x^2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from matplotlib.colors import to_rgb
from matplotlib.colors import LinearSegmentedColormap
from sklearn.model_selection import train_test_split



import torch
import matplotlib.pyplot as plt

# Parameters
n_samples = 300
batch_size = 10

subfolder = '1dexample'
import os
if not os.path.exists(subfolder):
    os.makedirs(subfolder)

# # Sample x from uniform distribution in [-1, 1]
# x = 2 * torch.rand(n_samples, 1) - 1  # shape (n_samples, 1), range [-1, 1]

# # Labels: y = x^2
# y = x ** 2  # shape (n_samples, 1)

# # Plot for verification
# plt.scatter(x.numpy(), y.numpy(), alpha=0.6)
# plt.xlabel("x")
# plt.ylabel("y = x^2")
# plt.title("Sampled Data from [-1, 1] with Labels x^2")
# plt.grid(True)
# plt.show()

def make_x2in1d(output_dim, n_samples = 100, plot = True, batch_size = batch_size, filename = None):
    """Generates xor
    """
    # Generate training data
    # set random seed for reproducibility
    seed = np.random.randint(1000)
    print(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    
    data = 2 * torch.rand(n_samples, 1) - 1  # shape (n_samples, 1), range [-1, 1]
    labels = data ** 2  # shape (n_samples, 1)
    # Generate outer ring points
    
    
    if plot:
                # Plot for verification
        plt.scatter(data.numpy(), labels.numpy(), alpha=0.6)
        plt.xlabel("x")
        plt.ylabel("y = x^2")
        plt.title("Sampled Data from [-1, 1] with Labels $x^2$")
        plt.grid(True)
        plt.show()
        
                # Save plot if filename provided
        if filename is not None:
            plt.savefig(f'{filename}.png', bbox_inches='tight', dpi=300)
            print(f'Plot saved as {filename}.png')
        
        plt.show()
    
    # Convert to tensors
    # data_tensor = torch.tensor(data, dtype=torch.float32)

    labels_tensor = torch.tensor(labels, dtype=torch.float32)


    data = torch.tensor(data, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.float32) 
    labels = torch.tensor(labels.reshape(-1, 1), dtype=torch.float32)
   
    # Create DataLoader for training data
    train_dataset = TensorDataset(data, labels)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
    return train_dataloader

train_loader = make_x2in1d(1, n_samples=n_samples)

## Plot function

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import io
import imageio
from matplotlib.colors import LinearSegmentedColormap

import plots.plots
from plots.plots import psi_manual
from models.training import train_model

# --- Aesthetic Palette Definitions ---
COLOR_DATA = '#34495e'      # Muted slate for background data
COLOR_MODEL = '#e63946'     # Vibrant crimson/coral for model predictions
COLOR_WEIGHT = '#2a9d8f'    # Deep teal for weights
COLOR_BIAS = '#f4a261'      # Warm sand for biases
COLOR_ZERO = '#d62828'      # Strong red for zero crossings
# Qualitative palette for multiple layers (repeats if layers > 7)
LAYER_COLORS = ['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76c24', '#e63946', '#8e44ad']

def apply_modern_styling(ax):
    """Helper function to apply clean, modern aesthetics to an axis."""
    # Light, dashed grid lines that don't distract from the data
    ax.grid(True, linestyle='--', alpha=0.5, color='#b0bec5', zorder=0)
    
    # Ensure all edges (spines) of the subplot are identical and visible
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#90a4ae')
        spine.set_linewidth(1.0)
        
    ax.tick_params(colors='#546e7a')
    ax.xaxis.label.set_color('#37474f')
    ax.yaxis.label.set_color('#37474f')
    ax.title.set_color('#263238')
    ax.title.set_fontweight('bold')

def plot_params(model, ax=None, y_lim=5):
    """Plots the weights and biases of the linear layers in a model."""
    linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
    n_layers = len(linear_layers)

    weights, biases = [], []
    for layer in linear_layers:
        weights.append(layer.weight.detach().cpu().numpy().squeeze())
        biases.append(layer.bias.detach().cpu().numpy().squeeze())

    if ax is None:
        fig, ax = plt.subplots(figsize=(2, 4))

    if n_layers == 0:
        ax.set_title("No Linear layers found")
        return

    apply_modern_styling(ax)
    layer_index = [i + 1 for i in range(n_layers)]
    
    # Plot biases
    ax.scatter(layer_index, biases, label='Bias', zorder=3, 
               marker='^', s=50, color=COLOR_BIAS, alpha=0.85, edgecolors='white', linewidths=0.5)
    # Plot weights
    ax.scatter(layer_index, weights, label='Weight', zorder=3, 
               marker='s', s=40, color=COLOR_WEIGHT, alpha=0.85, edgecolors='white', linewidths=0.5)
    
    ax.set_xlabel('Layer Index')
    ax.set_ylabel('Parameter Value')
    ax.set_title('Weights and Biases', pad=12)
    ax.set_ylim(-y_lim, y_lim)
    ax.set_xlim(0.5, n_layers + 0.5)
    ax.set_xticks(layer_index)
    
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), 
              ncol=2, bbox_transform=ax.transAxes, frameon=False)

    if ax is None:
        plt.tight_layout()
        plt.show()
            
def plot_eigvals_new(model, ax=None):
    model.eval()
    target_ax = ax if ax is not None else plt.gca()
    apply_modern_styling(target_ax)

    for layer in range(model.num_hidden + 1):
        func = lambda x: model.sub_model_new(x, from_layer=layer, to_layer=layer)
        interval = torch.linspace(-1, 1, 100)
        psi_values = np.zeros(100)
        
        for i, value in enumerate(interval):
            value = value.unsqueeze(0).unsqueeze(1) 
            psi_values[i] = psi_manual(value, func, output_type='eigvals_1d')

        # Use the custom color list, wrapping around if there are many layers
        line_color = LAYER_COLORS[layer % len(LAYER_COLORS)]
        target_ax.plot(interval.numpy(), psi_values, label=f'Layer {layer + 1}', 
                       color=line_color, linewidth=2, zorder=4, alpha=0.9)
        
        # --- Highlight Zero Crossings ---
        for i in range(len(psi_values) - 1):
            if psi_values[i] * psi_values[i+1] <= 0:
                if psi_values[i] == 0:
                    x_zero = interval[i].item()
                elif psi_values[i+1] == 0:
                    continue  
                else:
                    x0, x1 = interval[i].item(), interval[i+1].item()
                    y0, y1 = psi_values[i], psi_values[i+1]
                    x_zero = x0 - y0 * (x1 - x0) / (y1 - y0)
                
                target_ax.plot(x_zero, 0, marker='o', markerfacecolor='white', 
                               markeredgecolor=COLOR_ZERO, markersize=8, 
                               markeredgewidth=2, alpha=0.9, zorder=5)

    # Dashed prominent zero line
    target_ax.axhline(0, color='white', linewidth=2, zorder=2.5)
    target_ax.axhline(0, color='#2c3e50', linewidth=1.5, linestyle='--', alpha=0.6, zorder=3)

    if ax is None:
        target_ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9, frameon=False)
        target_ax.set_title('Eigenvalues', pad=12)
        target_ax.set_xlabel('Layer Input')
    else:
        # Better column logic for legend so it stays compact
        target_ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), 
                         ncol=3, bbox_transform=target_ax.transAxes, frameon=False, fontsize=9)
        target_ax.set_title('Eigenvalues', pad=12)
        target_ax.set_xlabel('Input')
        target_ax.set_box_aspect(1)

def subplots_model(model, epoch=None, filename=None, y_lim_params=5):
    x_plot = torch.linspace(-1, 1, 100).unsqueeze(1)
    y_plot = model(x_plot).detach().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(14, 5.5), gridspec_kw={'width_ratios': [7, 1.5, 7]})
    ax = axes[0]
    apply_modern_styling(ax)
    
    # Background training data (muted, slightly transparent)
    ax.scatter(train_loader.dataset.tensors[0].numpy(),
               train_loader.dataset.tensors[1].numpy(),
               label='Training Data', color=COLOR_DATA, alpha=0.4, s=15, edgecolors='none', zorder=2)
    # Vibrant foreground model line
    ax.plot(x_plot.numpy(), y_plot, label='ResNet Prediction', 
            color=COLOR_MODEL, linewidth=2, zorder=3)
    
    ax.set_xlabel('Input')
    ax.set_ylabel('Output')
    
    title_str = f'eps={model.skip_param}, delta={model.residual_param}'
    if epoch is not None:
        title_str += f', Epoch: {epoch}'
    ax.set_title(title_str, pad=12)
    
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), 
              ncol=2, bbox_transform=ax.transAxes, frameon=False)
    ax.set_box_aspect(1)
   
    plot_params(model, ax=axes[1], y_lim=y_lim_params)
    plot_eigvals_new(model, ax=axes[2])
 
    plt.subplots_adjust(
        left=0.05,
        right=0.9,
        wspace=-0.1,
        bottom=0.3,  # 30% of height reserved for legends
        top=0.9
    )
    
    if filename is not None:
        plt.savefig(f'{filename}.png', bbox_inches='tight', facecolor='white', dpi=300)
        print(f'Plot saved as {filename}.png')

def generate_gif(model, gif_name='last.gif', epochs_per_frame=10, num_frames=30, fps=5, loop='yes', lr=0.01):
    frames = [] 

    for i in range(num_frames):
        model, acc, lossess = train_model(
            model,
            train_loader,           
            train_loader,
            load_file=None,
            epochs=epochs_per_frame,
            lr=lr,
            early_stopping=False,
            cross_entropy=False,
            seed=None
        )

        subplots_model(model, epoch=i * epochs_per_frame)
        
        fig = plt.gcf()                         
        buf = io.BytesIO()                      
        fig.savefig(buf, format="png", dpi=300, facecolor='white', bbox_inches='tight') 
        buf.seek(0)                             

        frames.append(imageio.imread(buf))        
        plt.close(fig)                          

    loop_num = 0 if loop == 'no' else 1
    imageio.mimsave(gif_name, frames, fps=fps, loop=loop_num, subrectangles=False)
    print(f"Saved → {gif_name}")

# One layer experiments

In [ ]:
# to reload models.resnet module after changes without restarting the kernel
import importlib
import models.resnets
import models.training
importlib.reload(models.resnets) # Reload the module
importlib.reload(models.training) # Reload the module
from models.resnets import ResNet
from models.training import compute_accuracy, train_model, train_until_threshold, plot_loss_curve

# Model Params
input_dim = 1
hidden_dim = 1
num_hidden = 1 # number of hidden layers. The total network has additionl 2 layers: input to hidden and hidden to output
output_dim = 1
activation = 'tanh' #'relu' and 'tanh' are supported (currently only weights inside the hidden layers are supported)
input_layer = False #as simple as possible, no input layer
final_sigmoid = False # True supported with binary classification only

# Training Params
load_file = None
cross_entropy = False #True
batchnorm = False
num_epochs = 100


In [ ]:
mlp = ResNet(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, activation=activation, skip_param=0, final_sigmoid=final_sigmoid, input_layer=input_layer, batchnorm=batchnorm)

mlp, acc_base, losses_base = train_model(mlp,
    train_loader, train_loader, load_file = None, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)

plot_loss_curve(losses_base, title=f"Base Model Loss Curve", filename = 'ff1d')

subplots_model(mlp, epoch = None, filename= subfolder + '/mlp', y_lim_params=5)

In [ ]:
resnet_skip1 = ResNet(input_dim=input_dim, hidden_dim=hidden_dim, output_dim=output_dim, num_hidden=num_hidden, activation=activation, skip_param=1, final_sigmoid=final_sigmoid, input_layer=input_layer, batchnorm=batchnorm)

num_epochs = 300

resnet_skip1, acc_skip1, losses_skip1 = train_model(resnet_skip1,
    train_loader, train_loader, load_file = None, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)

plot_loss_curve(losses_base, title=f"Base Model Loss Curve", filename = 'ff1d')


In [ ]:

subplots_model(resnet_skip1, epoch = None, filename= subfolder + '/resnet_skip1', y_lim_params=5)

In [ ]:
for name, param in resnet_skip1.named_parameters():
    print(f"Name: {name} | Shape: {param.shape} | Requires Grad: {param.requires_grad}")
    
for name, param in resnet_skip1.named_parameters():
    print(f"--- {name} ---")
    print(param.data)
    print("\n")

In [ ]:
import io
import imageio      # v2 API keeps .mimsave unchanged
import matplotlib.pyplot as plt
import torch
 
# -----------------------------  CONFIG  ---------------------------------
gif_name = subfolder + "/skip0_training.gif"
fps = 1                    # 5 frames ≈ 200 ms between frames
frames = []                # collects RGB arrays for GIF
# ------------------------------------------------------------------------

hidden_dim   = 1
num_hidden   = 1
skip_param   = 0
num_frames = 20

model_skip0 = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, input_layer=input_layer)



generate_gif(model_skip0, gif_name = gif_name, num_frames = 1)
subplots_model(model_skip0)

In [ ]:
from IPython.display import Image, display

display(Image(filename=subfolder + "/skip0_training.gif", width=500))

In [ ]:
import io
import imageio      # v2 API keeps .mimsave unchanged
import matplotlib.pyplot as plt
import torch

# -----------------------------  CONFIG  ---------------------------------
gif_name = subfolder + "/skip1_training.gif"
fps = 5                    # 5 frames ≈ 200 ms between frames
frames = []                # collects RGB arrays for GIF
# ------------------------------------------------------------------------

hidden_dim   = 1
num_hidden   = 1
skip_param   = 1

model_skip1 = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, input_layer=input_layer)

# generate_gif(model_skip1, gif_name=gif_name)

In [ ]:
# display(Image(filename=subfolder + "/skip1_training.gif", width=600))

In [ ]:
skip_param = 1
hidden_dim   = 1
num_hidden   = 1

model_test = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

# generate_gif(model_test, gif_name = subfolder + 'test.gif', num_frames = 50)

In [ ]:
# display(Image(filename=subfolder + "test.gif", width=900))

# Two Layers

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

skip_param = 0
hidden_dim = 1
num_hidden = 2

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)


num_epochs = 500

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = None, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)

subplots_model(model, epoch = None, filename= subfolder + '/2l_eps0')
# generate_gif(model, gif_name = subfolder + '/test2.gif', num_frames = 50)
# display(Image(filename=subfolder + "/test2.gif", width=900))
# plot_eigvals(model)

In [ ]:
print(model)

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

skip_param = 1
residual_param = 1
hidden_dim = 1
num_hidden = 2

model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

num_epochs = 200

model, acc, lossess = train_model(model,
    train_loader, train_loader, load_file = None, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)

subplots_model(model, epoch = None, filename= subfolder + '/2l_eps1delta1')
# generate_gif(model, gif_name = subfolder + '/1d2l_eps1delta1.gif', num_frames = 50)
# display(Image(filename=subfolder + '/1d2l_eps1delta1.gif', width=900))
    

In [ ]:
model = ResNet(input_dim=input_dim,
                     hidden_dim=hidden_dim,
                     output_dim=output_dim,
                     num_hidden=num_hidden,
                     activation=activation,
                     skip_param=skip_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)

model, acc, losses = train_model(model,
    train_loader, train_loader, load_file = '1d2l_e1d1_best', epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)

model, acc, losses = train_model(model,
    train_loader, train_loader, load_file = None, epochs=num_epochs, lr=0.01, early_stopping = False, cross_entropy=False, seed = None)


subplots_model(model, epoch = None, filename= subfolder + '/2l_eps1delta1_best')

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

counter = 0

for skip in [1, 1, 1]:
    for sara in [1, 1, 1]:
        skip_param = skip
        residual_param = sara
        hidden_dim = 1
        num_hidden = 2

        model_running = ResNet(input_dim=input_dim,
                            hidden_dim=hidden_dim,
                            output_dim=output_dim,
                            num_hidden=num_hidden,
                            activation=activation,
                            skip_param=skip_param, residual_param=residual_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)
        gif_name = subfolder + '/running' + str(counter) + '.gif'
        generate_gif(model_running, gif_name = gif_name, fps = 2, num_frames = 50)
        counter += 1
        display(Image(filename=gif_name, width=900))
    

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet

counter = 0

for skip in [1, 1]:
    for sara in [1, 1]:
        skip_param = skip
        residual_param = sara
        hidden_dim = 1
        num_hidden = 3

        model_running = ResNet(input_dim=input_dim,
                            hidden_dim=hidden_dim,
                            output_dim=output_dim,
                            num_hidden=num_hidden,
                            activation=activation,
                            skip_param=skip_param, residual_param=residual_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)
        gif_name = subfolder + '/running' + str(counter) + '.gif'
        generate_gif(model_running, gif_name = gif_name, fps = 2, num_frames = 50)
        counter += 1
        display(Image(filename=gif_name, width=900))
    

In [ ]:
import models.resnets; importlib.reload(models.resnets)
from models.resnets import ResNet, ResidualBlock

counter = 0

for skip in [1]:
    for sara in [0.1]:
        skip_param = skip
        residual_param = sara
        hidden_dim = 1
        num_hidden = 2

        model_running = ResNet(input_dim=input_dim,
                            hidden_dim=hidden_dim,
                            output_dim=output_dim,
                            num_hidden=num_hidden,
                            activation=activation,
                            skip_param=skip_param, residual_param=residual_param, final_sigmoid=final_sigmoid, batchnorm=False, input_layer=input_layer)
        
        # for reproducibility (optional)

# find all ResidualBlocks inside the model
        blocks = [m for m in model_running.modules() if isinstance(m, ResidualBlock)]


        with torch.no_grad():
            # for block in blocks:
            #     nn.init.xavier_normal_(block.fc.weight, gain=1/residual_param)
        

            blocks[0].fc.weight[0] = -2/residual_param
            blocks[1].fc.weight[0] = -0.5/residual_param
    
            
        gif_name = subfolder + '/running' + str(counter) + '.gif'
        
        
        
        generate_gif(model_running, gif_name = gif_name, fps = 2, epochs_per_frame = 1, num_frames = 40, lr = 0.01)
        counter += 1
        display(Image(filename=gif_name, width=900))
    